<a href="https://colab.research.google.com/github/quasarx-snips/devjams_lunap/blob/main/Module_1/ResNet_Hazard_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip -q install -U segmentation-models-pytorch datasets huggingface_hub tqdm

import os, sys, random, time, math, glob, zipfile, requests
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm.auto import tqdm

SEED = 95 #dont ask why
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #if on gcolab set it from runtime then t4gpu
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("Setting to cuDNN...")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: None (Running on CPU)")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", device)




Setting to cuDNN...
GPU: Tesla T4
Python: 3.13.15
PyTorch: 2.11.0+cu128
Device: cuda


In [9]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CKPT_DIR = "/content/drive/MyDrive/lunap_hazard_checkpoints"
except Exception as e:
    CKPT_DIR = "/content/checkpoints"
    print("Drive mount skipped:", e)

os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint directory:", CKPT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint directory: /content/drive/MyDrive/lunap_hazard_checkpoints


In [10]:
from datasets import load_dataset

print("Loading crater dataset...")
crater_ds = load_dataset("gremlin97/crater_binary_segmentation")
print(crater_ds)
for split in ("train", "val", "test"):
    if split in crater_ds:
        print(f"{split:>5}: {len(crater_ds[split]):,} samples")


Loading crater dataset...
DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'width', 'height', 'class_labels'],
        num_rows: 3600
    })
    test: Dataset({
        features: ['image', 'mask', 'width', 'height', 'class_labels'],
        num_rows: 900
    })
    val: Dataset({
        features: ['image', 'mask', 'width', 'height', 'class_labels'],
        num_rows: 900
    })
})
train: 3,600 samples
  val: 900 samples
 test: 900 samples


In [12]:
import os
import zipfile
from pathlib import Path
import requests
from tqdm import tqdm

# Use local Colab SSD for heavy I/O to avoid Google Drive FUSE bottlenecks
DATA_ROOT = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)

DOORS_URL = "https://zenodo.org/records/7107409/files/DOORS.zip?download=1"
DOORS_ZIP = os.path.join(DATA_ROOT, "DOORS.zip")
DOORS_DIR = os.path.join(DATA_ROOT, "DOORS")

if not os.path.isdir(DOORS_DIR) or not any(Path(DOORS_DIR).rglob("*")):
    if not os.path.exists(DOORS_ZIP):
        print("Downloading DOORS (~2.7 GB) to local SSD...")
        with requests.get(DOORS_URL, stream=True, timeout=30) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(DOORS_ZIP, "wb") as f:
                with tqdm(total=total, unit="B", unit_scale=True, desc="DOORS.zip") as pbar:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

    print("Extracting DOORS (this may take a few minutes)...")
    os.makedirs(DOORS_DIR, exist_ok=True)
    with zipfile.ZipFile(DOORS_ZIP, "r") as z:
        members = z.infolist()
        with tqdm(total=len(members), desc="Extracting DOORS") as pbar:
            for member in members:
                z.extract(member, DOORS_DIR)
                pbar.update(1)
    os.remove(DOORS_ZIP)

print("DOORS ready at:", DOORS_DIR)

DOORS.zip: 100%|██████████| 2.70G/2.70G [14:05<00:00, 3.20MB/s]


Extracting DOORS (this may take a few minutes)...


Extracting DOORS: 100%|██████████| 477866/477866 [01:03<00:00, 7488.54it/s] 


DOORS ready at: /content/data/DOORS


In [13]:
from pathlib import Path
import os

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def image_files(folder):
    try:
        return [str(p) for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    except Exception:
        return []

doors_path = Path(DOORS_DIR)
print(f"Scanning {doors_path} for image and mask folders...")

all_img_dirs = []
all_mask_dirs = []

# Globally search every directory in the extracted tree
for p in doors_path.rglob("*"):
    if p.is_dir():
        name = p.name.lower()
        files = image_files(p)
        if len(files) > 0:
            if any(k in name for k in ["mask", "masks", "label", "labels", "gt", "seg"]):
                all_mask_dirs.append((str(p), len(files)))
            elif any(k in name for k in ["img", "image", "images"]):
                all_img_dirs.append((str(p), len(files)))

print("Found image directories:", len(all_img_dirs))
print("Found mask directories:", len(all_mask_dirs))

if not all_img_dirs or not all_mask_dirs:
    raise RuntimeError("DOORS dataset files are missing or incomplete. Try re-extracting locally to /content/data instead of Google Drive.")

# Sort to pick the largest available split
all_img_dirs.sort(key=lambda x: x[1], reverse=True)
all_mask_dirs.sort(key=lambda x: x[1], reverse=True)

BOULDER_IMG_DIR = all_img_dirs[0][0]
BOULDER_MASK_DIR = all_mask_dirs[0][0]

def norm_stem(path):
    s = Path(path).stem.lower()
    for token in ("_mask", "-mask", "_label", "-label", "_gt", "-gt", "_seg", "-seg"):
        s = s.replace(token, "")
    return s

img_paths = image_files(BOULDER_IMG_DIR)
mask_paths = image_files(BOULDER_MASK_DIR)

mask_by_key = {norm_stem(p): p for p in mask_paths}
pairs = [(p, mask_by_key[norm_stem(p)]) for p in img_paths if norm_stem(p) in mask_by_key]

if len(pairs) < 100:
    raise RuntimeError(f"Only {len(pairs)} valid image/mask pairs matched. Check if the dataset extracted correctly.")

print(f"Matched {len(pairs):,} boulder image/mask pairs")
print("Images:", BOULDER_IMG_DIR)
print("Masks :", BOULDER_MASK_DIR)
print("Example:", pairs[0])

Scanning /content/data/DOORS for image and mask folders...
Found image directories: 10
Found mask directories: 2
Matched 45,269 boulder image/mask pairs
Images: /content/data/DOORS/DOORS/Segmentation/DS1/DS/img
Masks : /content/data/DOORS/DOORS/Segmentation/DS1/DS/mask
Example: ('/content/data/DOORS/DOORS/Segmentation/DS1/DS/img/TR_015598.png', '/content/data/DOORS/DOORS/Segmentation/DS1/DS/mask/TR_015598.png')
